# Datamog in a Jupyter notebook

A Jupyter notebook is a great surface for working with Datamog interactively: you can keep declarations and rules in one cell, query results in the next, and have them render as pandas DataFrames you can chart, filter, or copy into a follow-up cell. Datamog ships a small Python package — `datamog-magic` — that hooks `bun run datamog --repl --json` into IPython as a `%%datamog` cell magic.

This notebook walks through using it on the Titanic CSV that the playground already exposes as a built-in example. Run cells with **Shift-Enter**.

## Install

**If you're in the repo's devcontainer**, the venv, the kernel, and `datamog-magic` itself are all pre-set-up. Run `jupyter lab --no-browser --ip=0.0.0.0` from a terminal, follow the forwarded URL, and pick **Python (datamog)** as the kernel — skip the rest of this section.

**Otherwise**, the magic shells out to a long-lived `datamog` REPL subprocess, so you need [Bun](https://bun.sh) on `PATH` and a checkout of this repo. Modern pip refuses to touch the system Python (PEP 668), so install into a virtualenv:

```bash
python3 -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\activate
pip install -e 'python/datamog-magic[pandas]' jupyter ipykernel matplotlib
python -m ipykernel install --user --name datamog --display-name "Python (datamog)"
```

Open this notebook with the **Python (datamog)** kernel — without the `ipykernel install` step, JupyterLab silently uses some other Python that doesn't have `datamog-magic` and `%load_ext` would fail with `ModuleNotFoundError`.

Sanity-check the kernel is the venv's Python:

In [ ]:
import sys
print(sys.executable)

The path should point inside a virtualenv, not the system Python — `.venv/bin/` for the manual install above, or `/opt/datamog-venv/bin/` in the repo's devcontainer (on Windows, the equivalent `Scripts\` location).

Now load the magic. You only need to do this once per kernel:

In [ ]:
%load_ext datamog_magic

## Hello, Datamog

Every Datamog cell starts with `%%datamog` on its very first line — that's the IPython *cell magic* header that tells Jupyter to send the whole cell to Datamog instead of trying to parse it as Python. Skip the header and you'll see a Python `SyntaxError` on the first `.`.

A tiny one-cell program:

In [ ]:
%%datamog
extensional greeting(message: string).
loud(M) :- greeting(M), length(M) > 0.

The cell declares an EDB and an IDB. Datamog knows the EDB has no rows because we didn't wire up a loader yet. Querying still works — it just returns an empty table:

In [ ]:
%%datamog
?- loud(M).

## Working with Titanic

The playground has a built-in Titanic example that pulls the CSV straight from pandas' GitHub raw URL. We can wire the same data into a notebook with `%datamog_init` (a *line* magic — one `%`, fits on a single line). We use the `gh:OWNER/REPO/PATH` shorthand, which Datamog expands to the `raw.githubusercontent.com` URL (the ref defaults to the repo's default branch):

In [ ]:
%datamog_init --backend sqlite --extensional passenger=gh:pandas-dev/pandas/doc/data/titanic.csv

`%datamog_init` shuts down any existing subprocess and queues a fresh one with the new flags. The next `%%datamog` cell spawns it.

Now declare the schema for the CSV and define a few aggregates:

In [ ]:
%%datamog
extensional passenger(
    PassengerId: integer,
    Survived:    integer,
    Pclass:      integer,
    Name:        string,
    Sex:         string,
    Age:         float?,
    SibSp:       integer,
    Parch:       integer,
    Ticket:      string,
    Fare:        float,
    Cabin:       string?,
    Embarked:    string?
).

survival_by_sex(Sex, avg(Survived), count(*)) :-
    passenger(_, Survived, _, _, Sex, _, _, _, _, _, _, _).

survival_by_class(Class, avg(Survived), count(*)) :-
    passenger(_, Survived, Class, _, _, _, _, _, _, _, _, _).

fare_by_class(Class, avg(Fare)) :-
    passenger(_, _, Class, _, _, _, _, _, _, Fare, _, _).

Queries run as soon as you press Shift-Enter:

In [ ]:
%%datamog
?- survival_by_sex(Sex, Rate, N).

In [ ]:
%%datamog
?- survival_by_class(Class, Rate, N).

In [ ]:
%%datamog
?- fare_by_class(Class, AverageFare).

Each result is a real pandas `DataFrame`, so anything you'd reach for afterwards — `.plot.bar()`, `.to_csv(...)`, joining against another DataFrame — Just Works on the displayed value.

If you want to keep the result around for follow-up cells, add `--df NAME` to the cell magic. The cell still renders the result, *and* binds it as `NAME` in the user namespace:

In [ ]:
%%datamog --df survival_df
?- survival_by_sex(Sex, Rate, N).

Now `survival_df` is a regular pandas DataFrame — chart it, filter it, join it against anything else in the kernel:

In [ ]:
survival_df.plot.bar(x='Sex', y='Rate', title='Titanic survival rate by sex')

If a cell has more than one query, `--df` binds the *last* result and prints a small warning. If it has none (just declarations or rules), the binding is a no-op and the magic tells you why.

## Multi-cell sessions

The magic keeps one long-lived subprocess across cells, so every `%%datamog` cell builds on the same accumulated session. The same rule the bare REPL enforces applies here: **a predicate's rule set is locked once committed**. Trying to add another rule for `survival_by_sex` in a later cell is rejected:

In [ ]:
%%datamog
survival_by_sex(Sex, max(Survived)) :-
    passenger(_, Survived, _, _, Sex, _, _, _, _, _, _, _).

Two ways out:

1. **Define every rule for one predicate in one cell.** This is the normal pattern — Datalog rules are usually grouped by head anyway.
2. **`%datamog_reset`** wipes all accumulated state (declarations, rules, EDB tables) and starts a fresh subprocess on the next cell. You'll lose loaded data, so you'll need to re-declare anything you want to keep.

In [ ]:
%datamog_reset

## Other line magics

| Magic | What it does |
| ----- | ------------ |
| `%datamog_init [flags]` | Shut down the existing subprocess (if any), reconfigure, and queue a fresh one. Flags: `--backend`, `--data-dir`, `--cwd`, `--cmd`, `--extensional name=source` (repeatable). |
| `%datamog_reset` | Send `:reset` to the REPL — discards all accumulated declarations / rules / data without restarting the subprocess. |
| `%datamog_close` | Terminate the subprocess. The next `%%datamog` cell will spawn a fresh one with the most recent `_init` config. |

## What's actually happening

`%%datamog` writes the cell's contents to the subprocess's stdin, followed by a blank line that signals "end of chunk." The CLI parses the chunk, applies it incrementally to its in-memory session, and emits one ndjson event per declaration / rule / query result / error on stdout. The magic reads those events until it sees a `done` sentinel, then renders them — DataFrames for results, red HTML for errors, terse status lines for declarations.

You can see the same wire format directly:

In [ ]:
!echo 'extensional p(x: integer).' | bun run datamog --repl --json --backend sqlite

This is the same channel a future Jupyter kernel could speak; the magic is the smallest possible adapter on top.

## Limits

This is `datamog-magic` v1. Known rough edges:

- **Predicate redefinition isn't supported.** The underlying `IncrementalSession` forbids extending a predicate's rule set across chunks; `:reset` is the only way back. A finer-grained "drop and recreate" mode could relax this.
- **One subprocess per kernel.** No multi-session support yet — every `%%datamog` cell shares the same backend.
- **Subprocess overhead.** Each cell pays one stdin / stdout round-trip. For tutorial-sized programs this is well under a second; for large EDB loads (millions of rows) the SQL backends still do the bulk of the work, but the round-trip is no longer free.

If you hit a bug or want a feature, file an issue at [github.com/max-schaefer/datamog/issues](https://github.com/max-schaefer/datamog/issues).